# 03.1 Sequence Basics

The goal of this notebook is to make the most basic representation of sequence data completely clear.

Before entering NLP or time-series modeling, you should at least understand:

- token
- token id
- vocabulary
- sequence length
- embedding
- padding
- mask

If these concepts are fuzzy, later `LSTM` and `Attention` notebooks will quickly become confusing.


## Learning Goals

After this notebook, you should be able to:

1. Understand how discrete tokens become integer ids.
2. Understand common shapes for sequence batches.
3. Use `nn.Embedding` to map token ids to embeddings.
4. Understand why padding is needed.
5. Build a padding mask.
6. Prepare the shapes needed for later `LSTM` and `Attention` notebooks.

In [ ]:
import torch
import torch.nn as nn

## Tokens, Vocabulary, and Id Mapping

A sequence often begins as strings or other discrete symbols.

A model cannot directly consume strings, so we usually map them to integer ids first.


In [ ]:
sentences = [
    ["i", "like", "pytorch"],
    ["you", "like", "deep", "learning"],
    ["i", "study"],
]

special_tokens = ["<pad>", "<unk>"]
vocab = special_tokens + sorted({token for sent in sentences for token in sent})
stoi = {token: idx for idx, token in enumerate(vocab)}
itos = {idx: token for token, idx in stoi.items()}

print("vocab =", vocab)
print("stoi =", stoi)
print("itos[2] =", itos[2])

In [ ]:
encoded_sentences = [[stoi.get(token, stoi["<unk>"]) for token in sent] for sent in sentences]

print("original sentences / original sentences:", sentences)
print("encoded sentences / encoded sentences:", encoded_sentences)

## Sequence Length

One of the most common dimensions in sequence modeling is the sequence length.

For example:

- `['i', 'like', 'pytorch']`  has length  3
- `['you', 'like', 'deep', 'learning']`  has length  4

The problem is that sequence lengths within the same batch are often different.


## Why Do We Need Padding?

Because a tensor requires a uniform shape within a batch, shorter sequences must be padded to the same length.

We usually pad with the `<pad>` token.


In [ ]:
pad_id = stoi["<pad>"]
max_len = max(len(seq) for seq in encoded_sentences)
padded_sentences = [seq + [pad_id] * (max_len - len(seq)) for seq in encoded_sentences]

batch_ids = torch.tensor(padded_sentences, dtype=torch.long)

print("max_len =", max_len)
print("padded_sentences =", padded_sentences)
print("batch_ids =\n", batch_ids)
print("batch_ids.shape =", batch_ids.shape)

`batch_ids.shape == (3, 4)` means:

- 3 samples in the batch
- each sample is padded to length 4

In many PyTorch sequence models, we often use `batch_first=True`, so the shape is written as:

- `(batch_size, seq_len)`
- `(batch_size, seq_len, embedding_dim)`

In [ ]:
# Exercise 1
# Given the three sequences below, pad them to the same length.

seqs = [[3, 5], [2, 4, 6, 7], [9]]
pad_id = 0

# max_len =
# padded =
# batch =
# print(batch)
# print(batch.shape)

In [ ]:
# Exercise 1 Reference Solution

seqs = [[3, 5], [2, 4, 6, 7], [9]]
pad_id = 0
max_len = max(len(seq) for seq in seqs)
padded = [seq + [pad_id] * (max_len - len(seq)) for seq in seqs]
batch = torch.tensor(padded, dtype=torch.long)
print(batch)
print(batch.shape)

## 4. `nn.Embedding`

A token id is only an index; it does not carry semantic meaning by itself.

The role of an embedding is to map each token id to a dense vector.


In [ ]:
vocab_size = len(vocab)
embedding_dim = 6
embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

embedded = embedding(batch_ids)

print("batch_ids.shape =", batch_ids.shape)
print("embedded.shape =", embedded.shape)

Here the shape changes from `(batch_size, seq_len)` to `(batch_size, seq_len, embedding_dim)`.

That means every token becomes a vector.


In [ ]:
print("batch_ids[0] =", batch_ids[0])
print("embedded[0].shape =", embedded[0].shape)
print("embedded[0] =\n", embedded[0])

In [ ]:
# Exercise 2
# Build an embedding layer with vocab size 20 and embedding dimension 8.
# Then feed it a token-id batch of shape (4, 5) and print the output shape.

# emb =
# batch =
# out =
# print(out.shape)

In [ ]:
# Exercise 2 Reference Solution

emb = nn.Embedding(20, 8)
batch = torch.randint(low=0, high=20, size=(4, 5))
out = emb(batch)
print(out.shape)

## 5. Padding Mask

Although padding makes the batch shape uniform, `<pad>` is not real content.

So many sequence models need to know which positions are padding.

That is the role of a padding mask.


In [ ]:
padding_mask = batch_ids == pad_id

print("batch_ids =\n", batch_ids)
print("padding_mask =\n", padding_mask)
print("padding_mask.shape =", padding_mask.shape)

Here `True` means that the position is padding.

This kind of mask becomes very important in the later `Attention` notebook.


In [ ]:
# Exercise 3
# Given the batch ids below, construct the padding mask.

batch = torch.tensor([
    [4, 5, 0, 0],
    [1, 2, 3, 0],
    [7, 8, 9, 1],
])
pad_id = 0

# mask =
# print(mask)

In [ ]:
# Exercise 3 Reference Solution

batch = torch.tensor([
    [4, 5, 0, 0],
    [1, 2, 3, 0],
    [7, 8, 9, 1],
])
pad_id = 0
mask = batch == pad_id
print(mask)

## Common Shape Reference Table

You should get comfortable with this set of shapes as early as possible:

- token ids: `(batch_size, seq_len)`
- embeddings: `(batch_size, seq_len, embedding_dim)`
- padding mask: `(batch_size, seq_len)`

Later notebooks mostly add new semantics on top of these basic dimensions.


## Summary

The core of this notebook is not the number of APIs, but fully understanding the input format of sequence data.

You should now be able to answer:

1. Why are tokens often mapped to integer ids first?
2. Why do we need padding within the same batch?
3. Why does an `Embedding` turn `(B, T)` into `(B, T, D)`?
4. What is the role of a padding mask?

Suggested next step:

- Move to the `LSTM` notebook and see how these sequence tensors are processed by a recurrent model.